# Modeling — Advertisement Click on Ad

**Course:** P4AI-DS (CO3135) — HCMUT
**Assignment 1:** End-to-end tabular ML pipeline — Phases B + C

**Reads with:** [`STUDY_GUIDE.md`](STUDY_GUIDE.md) (Phases B & C) and [`eda_tabular.ipynb`](eda_tabular.ipynb) (EDA findings).

## What this notebook produces

1. A stratified 80/20 train/test split, saved to `../plots_data/splits/` for reuse by the evaluation notebook.
2. A feature-engineering + preprocessing pipeline implemented as an `sklearn` `ColumnTransformer` that turns the raw 9-column dataframe into a leak-proof model-ready matrix.
3. A **3-model comparison** via stratified 5-fold CV: Logistic Regression (linear baseline), Random Forest (bagged nonlinear), Histogram Gradient Boosting (boosted nonlinear).
4. `RandomizedSearchCV` hyperparameter tuning on the CV winner.
5. The final fitted `Pipeline` serialized to `artifacts/best_model.joblib` for the evaluation notebook.

## Design contract with the EDA notebook

| EDA finding | Pipeline action taken below |
|---|---|
| 0 missing, 0 duplicates, 50/50 balance | No imputer logically needed, but wired as safety; no class weights; no SMOTE |
| `Ad Topic Line` (cardinality 1.00), `City` (0.97) | Dropped |
| `Country` (237 levels) | Target-encoded with out-of-fold CV inside the pipeline |
| `Timestamp` (unique datetimes) | Parsed to `hour`, `dayofweek`, `month` + cyclic sin/cos |
| Wide scale differences (`Income` ~10^4 vs `Male` in {0,1}) | `StandardScaler` on numeric branch of the linear pipeline; passthrough on trees |
| Moderate Daily Time ↔ Internet Usage corr (r ≈ 0.52) | L2 regularization in Logistic Regression (`C` tuned) |
| Strong roughly-linear class separation | Logistic Regression is a genuine competitor, not a strawman |

## 0. Setup

**Why a dedicated setup cell.** Reproducibility requires three things pinned at the top of every notebook: the RNG seed, the library versions, and the filesystem paths. Anyone running this notebook should see identical numbers.

**Dependencies:** `pandas`, `numpy`, `scikit-learn>=1.3` (required for `TargetEncoder`), `matplotlib`, `seaborn`, `joblib`. Optional: `xgboost>=2.0` (we fall back to `HistGradientBoostingClassifier` if unavailable).

In [6]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import sklearn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
warnings.filterwarnings("ignore")

DATA_PATH = Path("../data/advertising.csv")
ARTIFACTS_DIR = Path("./artifacts")
SPLITS_DIR = Path("../plots_data/splits")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

assert sklearn.__version__ >= "1.3", (
    f"sklearn>=1.3 is required for TargetEncoder; got {sklearn.__version__}. "
    f"Run: pip install -U scikit-learn"
)

print(f"scikit-learn: {sklearn.__version__}")
print(f"numpy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")
print(f"artifacts:    {ARTIFACTS_DIR.resolve()}")
print(f"splits:       {SPLITS_DIR.resolve()}")

scikit-learn: 1.8.0
numpy:        2.4.3
pandas:       3.0.2
artifacts:    C:\Users\Admin\Documents\Code\HCMUTcode\252code\P4AI&DS\btl\btl1\EDA-Techniques-for-AI-and-DS\notebooks\artifacts
splits:       C:\Users\Admin\Documents\Code\HCMUTcode\252code\P4AI&DS\btl\btl1\EDA-Techniques-for-AI-and-DS\plots_data\splits


## 1. Load data and reapply EDA findings

**Why reload here.** The modeling notebook must be runnable end-to-end without first running the EDA notebook. Anything we learned in EDA is re-encoded as a **contract** (column lists below), not as runtime state.

**Contract column lists.**
- `TARGET = "Clicked on Ad"` — binary (0/1), 50/50 balanced.
- `NUMERIC_COLS` — the four continuous behavioral/demographic predictors.
- `BINARY_COLS` — `Male` (already 0/1).
- `CATEGORICAL_MID_COLS` — `Country` (237 levels, target-encoded).
- `TIMESTAMP_COL` — parsed into hour/dow/month + cyclic encoding.
- `DROP_COLS` — identifiers (`Ad Topic Line`, `City`).

In [7]:
TARGET = "Clicked on Ad"
NUMERIC_COLS = [
    "Daily Time Spent on Site",
    "Age",
    "Area Income",
    "Daily Internet Usage",
]
BINARY_COLS = ["Male"]
CATEGORICAL_MID_COLS = ["Country"]
TIMESTAMP_COL = "Timestamp"
DROP_COLS = ["Ad Topic Line", "City"]

df = pd.read_csv(DATA_PATH)
df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL])

assert df[TARGET].isna().sum() == 0, "Unexpected missing target."
assert set(df[TARGET].unique()) == {0, 1}, "Target must be binary 0/1."

print(f"Shape:           {df.shape}")
print(f"Target balance:  {df[TARGET].value_counts(normalize=True).to_dict()}")
print(f"Missing cells:   {int(df.isna().sum().sum())}")
print(f"Duplicate rows:  {int(df.duplicated().sum())}")
df.head()

Shape:           (1000, 10)
Target balance:  {0: 0.5, 1: 0.5}
Missing cells:   0
Duplicate rows:  0


,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,City,Male,Country,Timestamp,Clicked on Ad
0,68.95,35,61833.90,256.09,Cloned 5thgeneration orchestration,Wrightburgh,0,Tunisia,2016-03-27 00:53:11,0
1,80.23,31,68441.85,193.77,Monitored national standardization,West Jodi,1,Nauru,2016-04-04 01:39:02,0
2,69.47,26,59785.94,236.50,Organic bottom-line service-desk,Davidton,0,San Marino,2016-03-13 20:35:42,0
3,74.15,29,54806.18,245.89,Triple-buffered reciprocal time-frame,West Terrifurt,1,Italy,2016-01-10 02:31:19,0
4,68.37,35,73889.99,225.58,Robust logistical utilization,South Manuel,0,Iceland,2016-06-03 03:36:18,0


## 2. Stratified train/test split

**Why stratify even at 50/50.** Stratification reduces the *variance* of fold estimates; random draws on 1,000 rows can easily give 48/52 splits, and that noise compounds over 5 folds. Stratify always.

**Why 80/20.** 200 test points give a ~±6 pp Wilson 95 % CI on accuracy — tight enough to be meaningful. 90/10 would give ~±9 pp, too noisy.

**Why fix `random_state=42`.** Anyone re-running this notebook must get the same split and therefore the same final metrics.

**Leakage note.** The split happens *before* any transformation that learns from data (scaler, target encoder). Those live inside the Pipeline and are refit per CV fold.

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f"Train: {X_train.shape}, class balance: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Test:  {X_test.shape}, class balance: {y_test.value_counts(normalize=True).to_dict()}")

X_train.to_parquet(SPLITS_DIR / "X_train.parquet", index=False)
X_test.to_parquet(SPLITS_DIR / "X_test.parquet", index=False)
y_train.to_frame().to_parquet(SPLITS_DIR / "y_train.parquet", index=False)
y_test.to_frame().to_parquet(SPLITS_DIR / "y_test.parquet", index=False)
print(f"\nSplits saved to {SPLITS_DIR.resolve()}")

Train: (800, 9), class balance: {1: 0.5, 0: 0.5}
Test:  (200, 9), class balance: {1: 0.5, 0: 0.5}


ArrowKeyError: A type extension with name pandas.period already defined

## 3. Timestamp feature engineering

**Why engineer `Timestamp` instead of dropping it.** The column is unique (1,000 distinct datetimes) so it cannot be used raw — that would be an identifier. But the *components* (hour, day of week, month) may carry signal about ad engagement patterns.

**Why cyclic sin/cos encoding.**
- Raw integer `hour` tells a linear model that 23 and 0 are 23 apart — wrong, they are adjacent.
- `hour_sin = sin(2π · h / 24)`, `hour_cos = cos(2π · h / 24)` preserves cyclic adjacency.
- Trees do not strictly need sin/cos (they can split anywhere) but it is harmless.

**Why a custom `FunctionTransformer`.** It is deterministic and stateless, so it does not need to "fit" anything — it can live inside the pipeline to guarantee the timestamp logic is always applied identically.

In [10]:
from sklearn.preprocessing import FunctionTransformer


def extract_timestamp_features(X: pd.DataFrame) -> pd.DataFrame:
    """Parse Timestamp into hour, dayofweek, month, plus cyclic sin/cos of hour.

    Deterministic: does not learn anything from the data.
    Returns a dataframe with 5 numeric columns so downstream transformers see numbers.
    """
    ts = pd.to_datetime(X[TIMESTAMP_COL])
    hour = ts.dt.hour
    out = pd.DataFrame(
        {
            "ts_hour": hour.astype(float),
            "ts_dayofweek": ts.dt.dayofweek.astype(float),
            "ts_month": ts.dt.month.astype(float),
            "ts_hour_sin": np.sin(2 * np.pi * hour / 24.0),
            "ts_hour_cos": np.cos(2 * np.pi * hour / 24.0),
        },
        index=X.index,
    )
    return out


TIMESTAMP_FEATURES = ["ts_hour", "ts_dayofweek", "ts_month", "ts_hour_sin", "ts_hour_cos"]


def _ts_feature_names(_transformer, _input_features):
    return TIMESTAMP_FEATURES


timestamp_transformer = FunctionTransformer(
    extract_timestamp_features,
    validate=False,
    feature_names_out=_ts_feature_names,
)

demo = extract_timestamp_features(X_train.head(3))
print("Sample timestamp features:")
demo

Sample timestamp features:


,ts_hour,ts_dayofweek,ts_month,ts_hour_sin,ts_hour_cos
747,0.0,4.0,1.0,0.000000,1.0
586,20.0,1.0,1.0,-0.866025,0.5
519,16.0,5.0,6.0,-0.866025,-0.5


## 4. Preprocessing pipelines

We build **two** `ColumnTransformer`s so each model sees the representation that suits it.

| Branch | Linear / SVM / KNN pipeline | Tree / boosting pipeline |
|---|---|---|
| `NUMERIC_COLS` | median impute → `StandardScaler` | median impute only (scale-invariant) |
| `BINARY_COLS` (`Male`) | passthrough | passthrough |
| `CATEGORICAL_MID_COLS` (`Country`) | `TargetEncoder(cv=5)` | same |
| `TIMESTAMP_COL` | extract → `StandardScaler` on the 5 engineered columns | extract only |
| `DROP_COLS` | dropped | dropped |

**Why `TargetEncoder` over one-hot for `Country`.** 237 levels × 1,000 rows would create 236 sparse columns each with tiny support — a textbook curse-of-dimensionality trap. `TargetEncoder` maps each level to its smoothed out-of-fold mean target, so the representation is a single numeric column per category. The `cv=5` argument computes encodings on hold-out folds during training to prevent target leakage.

**Why two pipelines and not one.** Scaling wastes compute on trees and can slightly hurt split quality. Separating them is one line of code and keeps the comparison fair.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder


def build_numeric_pipe(scale: bool) -> Pipeline:
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps=steps)


def build_timestamp_pipe(scale: bool) -> Pipeline:
    steps = [
        (
            "extract",
            FunctionTransformer(
                extract_timestamp_features,
                validate=False,
                feature_names_out=_ts_feature_names,
            ),
        )
    ]
    if scale:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps=steps)


def build_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", build_numeric_pipe(scale=scale_numeric), NUMERIC_COLS),
            ("bin", "passthrough", BINARY_COLS),
            (
                "cat_country",
                TargetEncoder(target_type="binary", smooth="auto", cv=5, random_state=SEED),
                CATEGORICAL_MID_COLS,
            ),
            ("ts", build_timestamp_pipe(scale=scale_numeric), [TIMESTAMP_COL]),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


preprocessor_linear = build_preprocessor(scale_numeric=True)
preprocessor_tree = build_preprocessor(scale_numeric=False)

sample_out = preprocessor_linear.fit_transform(X_train, y_train)
print(f"Linear-pipe output shape:  {sample_out.shape}  (from raw {X_train.shape})")
print(f"Dropped columns:           {DROP_COLS}")
print(f"Feature names (after CT):  {list(preprocessor_linear.get_feature_names_out())}")

Linear-pipe output shape:  (800, 11)  (from raw (800, 9))
Dropped columns:           ['Ad Topic Line', 'City']
Feature names (after CT):  ['Daily Time Spent on Site', 'Age', 'Area Income', 'Daily Internet Usage', 'Male', 'Country', 'ts_hour', 'ts_dayofweek', 'ts_month', 'ts_hour_sin', 'ts_hour_cos']


## 5. Model zoo — three families, three learning biases

| Model | Learning bias | Scaling needed | Expected behavior here |
|---|---|---|---|
| **Logistic Regression** (L2) | Assumes log-odds are linear in features | Yes | Strong baseline; EDA showed near-linear target separation |
| **Random Forest** | Averages many decorrelated deep trees; captures interactions | No | Flexible; may overfit n=1,000 unless `max_depth` is capped |
| **Histogram Gradient Boosting** | Boosted shallow trees; strong regularization via `learning_rate` + early stopping | No | Usually the best tabular baseline; similar to XGBoost / LightGBM |

**Why HistGradientBoosting (HGB) over XGBoost by default.** HGB is built into `sklearn` (no extra dep), supports `class_weight`, handles missing values natively, and is within 1–2 pp of XGBoost on most tabular tasks. Swap in `XGBClassifier` if you want — the pipeline does not care.

**Expected ranking on this data.** LR ≈ HGB > RF. Why: the EDA showed nearly linear class separation on the two strongest features; LR can exploit that directly; HGB matches via shallow boosted trees; RF usually wastes capacity on deep trees that memorize on n=1,000.

In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "LogisticRegression": Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(scale_numeric=True)),
            ("clf", LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)),
        ]
    ),
    "RandomForest": Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(scale_numeric=False)),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=2,
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "HistGradientBoosting": Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(scale_numeric=False)),
            (
                "clf",
                HistGradientBoostingClassifier(
                    learning_rate=0.1,
                    max_iter=300,
                    max_depth=None,
                    max_leaf_nodes=31,
                    min_samples_leaf=20,
                    l2_regularization=0.0,
                    early_stopping=True,
                    random_state=SEED,
                ),
            ),
        ]
    ),
}

print("Pipelines built:")
for name in models:
    print(f"  - {name}")

Pipelines built:
  - LogisticRegression
  - RandomForest
  - HistGradientBoosting


## 6. Cross-validated comparison

**Why 5-fold stratified.** At n_train = 800, 5 folds mean 160 validation rows per fold — enough to stabilize the metric. 10-fold would give 80 rows per fold and inflate variance.

**Why these metrics.**
- `roc_auc` — threshold-free ranking quality (primary for the winner selection).
- `average_precision` (PR-AUC) — robust to class imbalance; sanity check.
- `accuracy`, `f1` — default-threshold performance.
- `neg_log_loss` — probability quality (lower is better; we negate so higher is better).

**Why `return_train_score=True`.** The train–validation gap is the variance diagnostic. If a model has train AUC = 1.00 and val AUC = 0.90, it is overfitting and needs more regularization.

In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = ["accuracy", "roc_auc", "average_precision", "f1", "neg_log_loss"]

cv_rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING,
        return_train_score=True,
        n_jobs=-1,
    )
    row = {"model": name}
    for metric in SCORING:
        val = scores[f"test_{metric}"]
        train = scores[f"train_{metric}"]
        row[f"{metric}_val_mean"] = np.mean(val)
        row[f"{metric}_val_std"] = np.std(val)
        row[f"{metric}_train_mean"] = np.mean(train)
    cv_rows.append(row)

cv_df = pd.DataFrame(cv_rows).set_index("model")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
cv_df[[c for c in cv_df.columns if c.endswith("val_mean") or c.endswith("val_std")]]

,accuracy_val_mean,accuracy_val_std,roc_auc_val_mean,roc_auc_val_std,average_precision_val_mean,average_precision_val_std,f1_val_mean,f1_val_std,neg_log_loss_val_mean,neg_log_loss_val_std
model,,,,,,,,,,
LogisticRegression,0.9662,0.0140,0.9883,0.0043,0.9918,0.0023,0.9658,0.0143,-0.1059,0.0181
RandomForest,0.9575,0.0203,0.9911,0.0048,0.9929,0.0034,0.9580,0.0192,-0.1387,0.0216
HistGradientBoosting,0.9563,0.0112,0.9888,0.0043,0.9913,0.0031,0.9561,0.0114,-0.1311,0.0223


In [14]:
summary = pd.DataFrame(
    {
        "roc_auc (val)": cv_df["roc_auc_val_mean"].map(lambda v: f"{v:.4f}")
        + " ± "
        + cv_df["roc_auc_val_std"].map(lambda v: f"{v:.4f}"),
        "pr_auc (val)": cv_df["average_precision_val_mean"].map(lambda v: f"{v:.4f}")
        + " ± "
        + cv_df["average_precision_val_std"].map(lambda v: f"{v:.4f}"),
        "accuracy (val)": cv_df["accuracy_val_mean"].map(lambda v: f"{v:.4f}")
        + " ± "
        + cv_df["accuracy_val_std"].map(lambda v: f"{v:.4f}"),
        "f1 (val)": cv_df["f1_val_mean"].map(lambda v: f"{v:.4f}")
        + " ± "
        + cv_df["f1_val_std"].map(lambda v: f"{v:.4f}"),
        "log_loss (val)": (-cv_df["neg_log_loss_val_mean"]).map(lambda v: f"{v:.4f}"),
        "train_roc_auc": cv_df["roc_auc_train_mean"].map(lambda v: f"{v:.4f}"),
        "overfit_gap_auc": (
            cv_df["roc_auc_train_mean"] - cv_df["roc_auc_val_mean"]
        ).map(lambda v: f"{v:+.4f}"),
    }
)

summary.to_csv(ARTIFACTS_DIR / "cv_comparison.csv")
print("Saved:", ARTIFACTS_DIR / "cv_comparison.csv")
summary

Saved: artifacts\cv_comparison.csv


,roc_auc (val),pr_auc (val),accuracy (val),f1 (val),log_loss (val),train_roc_auc,overfit_gap_auc
model,,,,,,,
LogisticRegression,0.9883 ± 0.0043,0.9918 ± 0.0023,0.9662 ± 0.0140,0.9658 ± 0.0143,0.1059,0.9924,+0.0041
RandomForest,0.9911 ± 0.0048,0.9929 ± 0.0034,0.9575 ± 0.0203,0.9580 ± 0.0192,0.1387,0.9998,+0.0087
HistGradientBoosting,0.9888 ± 0.0043,0.9913 ± 0.0031,0.9563 ± 0.0112,0.9561 ± 0.0114,0.1311,0.9992,+0.0103


## 7. Pick the winner

**Selection rule.** Highest mean CV ROC-AUC, breaking ties by **lowest overfit gap** (train AUC − val AUC) — we prefer a model whose CV estimate is more trustworthy.

**Why ROC-AUC as the tie-breaker metric.** It is threshold-free, so the ranking is not affected by our (later) threshold choice. We separately track log-loss for probability quality and will calibrate in the evaluation notebook.

In [15]:
ranking = cv_df[["roc_auc_val_mean", "roc_auc_train_mean"]].copy()
ranking["gap"] = ranking["roc_auc_train_mean"] - ranking["roc_auc_val_mean"]
ranking = ranking.sort_values(["roc_auc_val_mean", "gap"], ascending=[False, True])
print("CV ranking (by val ROC-AUC, tie-break by smaller overfit gap):")
print(ranking)

winner_name = ranking.index[0]
winner_pipe = models[winner_name]
print(f"\nWinner: {winner_name}")

CV ranking (by val ROC-AUC, tie-break by smaller overfit gap):
                      roc_auc_val_mean  roc_auc_train_mean    gap
model                                                            
RandomForest                    0.9911              0.9998 0.0087
HistGradientBoosting            0.9888              0.9992 0.0103
LogisticRegression              0.9883              0.9924 0.0041

Winner: RandomForest


## 8. Hyperparameter tuning on the winner

**Why `RandomizedSearchCV` over `GridSearchCV`.** Grid spends budget on axis-aligned combos; random explores the space more uniformly and scales better when hyperparameter count grows. At `n_iter=40`, we typically get within 1 % of the grid-best on a 4-D hyperparameter space.

**Why tune *after* model selection, not before.** Compute budget. Tuning all three families and then selecting is defensible and more rigorous — but for a learning project, tuning the winner is ~3× cheaper and the relative ranking at default settings is usually stable. Document the choice.

**What we tune per model family.** The hyperparameters that move the needle, per `STUDY_GUIDE.md §4.C2`:
- LR: `C` (inverse L2 strength), `penalty`, `solver`.
- RF: `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`.
- HGB: `learning_rate`, `max_iter`, `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization`.

In [16]:
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV

search_spaces = {
    "LogisticRegression": {
        "clf__C": loguniform(1e-3, 1e2),
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs", "liblinear"],
    },
    "RandomForest": {
        "clf__n_estimators": randint(200, 800),
        "clf__max_depth": [None, 4, 6, 8, 12, 16],
        "clf__min_samples_leaf": randint(1, 10),
        "clf__max_features": ["sqrt", "log2", 0.5, 0.8],
    },
    "HistGradientBoosting": {
        "clf__learning_rate": loguniform(0.01, 0.3),
        "clf__max_iter": randint(100, 600),
        "clf__max_leaf_nodes": randint(15, 63),
        "clf__min_samples_leaf": randint(10, 60),
        "clf__l2_regularization": uniform(0.0, 1.0),
    },
}

search = RandomizedSearchCV(
    estimator=winner_pipe,
    param_distributions=search_spaces[winner_name],
    n_iter=40,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    random_state=SEED,
    refit=True,
    verbose=1,
    return_train_score=True,
)
search.fit(X_train, y_train)

print(f"\nBest CV ROC-AUC: {search.best_score_:.4f}")
print(f"Best params:     {search.best_params_}")

Fitting 5 folds for each of 40 candidates, totalling 200 fits

Best CV ROC-AUC: 0.9915
Best params:     {'clf__max_depth': None, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 587}


In [17]:
cv_results = pd.DataFrame(search.cv_results_)
top = (
    cv_results[["mean_test_score", "std_test_score", "mean_train_score", "params"]]
    .sort_values("mean_test_score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top.to_csv(ARTIFACTS_DIR / "random_search_top10.csv", index=False)
print("Top-10 hyperparameter trials saved.")
top

Top-10 hyperparameter trials saved.


,mean_test_score,std_test_score,mean_train_score,params
0,0.9915,0.0039,0.9998,"{'clf__max_depth': None, 'clf__max_features': ..."
1,0.9912,0.0050,0.9987,"{'clf__max_depth': 12, 'clf__max_features': 'l..."
2,0.9911,0.0046,0.9975,"{'clf__max_depth': 6, 'clf__max_features': 'lo..."
3,0.9910,0.0051,0.9988,"{'clf__max_depth': 16, 'clf__max_features': 's..."
4,0.9909,0.0048,0.9976,"{'clf__max_depth': 6, 'clf__max_features': 'sq..."
5,0.9908,0.0043,0.9985,"{'clf__max_depth': 6, 'clf__max_features': 0.5..."
6,0.9908,0.0047,0.9999,"{'clf__max_depth': 8, 'clf__max_features': 'sq..."
7,0.9908,0.0047,0.9994,"{'clf__max_depth': 6, 'clf__max_features': 0.5..."
8,0.9907,0.0046,0.9994,"{'clf__max_depth': 6, 'clf__max_features': 0.5..."
9,0.9907,0.0046,0.9981,"{'clf__max_depth': None, 'clf__max_features': ..."


## 9. Persist the best model

The `search` object refit on the full training set at the best hyperparameters (because `refit=True`). That refitted pipeline is `search.best_estimator_`. We:

1. Serialize the whole pipeline (preprocessor + classifier) with `joblib.dump` so the evaluation notebook can `load` it.
2. Also save a metadata JSON with the winning model name, best params, and CV scores for traceability.

**Why serialize the whole Pipeline, not just the classifier.** Deployment must use exactly the same preprocessing (same target encoding, same scaler fit) that was used during training. A raw classifier without its preprocessor is not deployable.

In [18]:
import json

best_pipeline = search.best_estimator_
joblib.dump(best_pipeline, ARTIFACTS_DIR / "best_model.joblib")

metadata = {
    "winner": winner_name,
    "best_params": {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
                    for k, v in search.best_params_.items()},
    "best_cv_roc_auc": float(search.best_score_),
    "cv_comparison": summary.to_dict(orient="index"),
    "random_state": SEED,
    "sklearn_version": sklearn.__version__,
    "feature_columns": {
        "numeric": NUMERIC_COLS,
        "binary": BINARY_COLS,
        "categorical_mid": CATEGORICAL_MID_COLS,
        "timestamp": TIMESTAMP_COL,
        "dropped": DROP_COLS,
    },
}
with open(ARTIFACTS_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Saved: {ARTIFACTS_DIR / 'best_model.joblib'}")
print(f"Saved: {ARTIFACTS_DIR / 'model_metadata.json'}")
print(f"\nFinal winner:   {winner_name}")
print(f"Best CV ROC-AUC: {search.best_score_:.4f}")

Saved: artifacts\best_model.joblib
Saved: artifacts\model_metadata.json

Final winner:   RandomForest
Best CV ROC-AUC: 0.9915


## 10. What we just produced

```
artifacts/
├── best_model.joblib         # the whole Pipeline (preprocessing + classifier)
├── model_metadata.json       # winner name, best params, CV scores
├── cv_comparison.csv         # 3-model side-by-side comparison
└── random_search_top10.csv   # top-10 tuning trials

../plots_data/splits/
├── X_train.parquet
├── X_test.parquet
├── y_train.parquet
└── y_test.parquet
```

**Next stop:** open [`evaluation_tabular.ipynb`](evaluation_tabular.ipynb) which loads all of the above and produces:
- Held-out metrics (accuracy, ROC-AUC, PR-AUC, F1, log-loss, Brier).
- ROC, PR, and calibration curves.
- Threshold tuning with a cost-matrix framing.
- Learning curve diagnostics.
- Permutation importance + SHAP explanations.
- Error slice analysis.
- A one-page final report card.